In [1]:
import pandas as pd
import numpy as np
import re
import csv
import sys
import time
import os
import psutil
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk

nltk.download('stopwords', quiet=True)

# ── Constants ────────────────────────────────────────────────────────────────

STOP_WORDS = set(stopwords.words("english"))
STEMMER    = PorterStemmer()

URL_RE    = re.compile(r"\bhttps?://[^\s]+|www\.[^\s]+")
EMAIL_RE  = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}")
DATE_RE   = re.compile(r"\b(\d{4}-\d{2}-\d{2}|\d{1,2}[/-]\d{1,2}[/-]\d{2,4})\b")
NUM_RE    = re.compile(r"\d+")
TOKEN_RE  = re.compile(r"\b[a-z]+\b")


# ── Memory Utility ────────────────────────────────────────────────────────────

def print_memory():
    """Print current Python process RAM usage."""
    process = psutil.Process(os.getpid())
    used = process.memory_info().rss / 1e9
    available = psutil.virtual_memory().available / 1e9
    print(f"  RAM — Python: {used:.1f} GB | Available: {available:.1f} GB")


# ── Data Loading ─────────────────────────────────────────────────────────────

def load_data(path):
    """Load CSV from a URL or filepath, handling large field sizes."""
    csv.field_size_limit(sys.maxsize)
    return pd.read_csv(path, encoding="utf-8", engine="python")


# ── Cleaning ─────────────────────────────────────────────────────────────────

def clean_content(text):
    """
    Clean a single string: lowercase, strip, replace URLs/emails/dates/numbers.
    Caps article length at 50,000 characters to avoid monster articles.
    """
    if not isinstance(text, str):
        return ""
    text = text[:50000]
    text = text.lower().strip()
    text = URL_RE.sub("<URL>", text)
    text = EMAIL_RE.sub("<EMAIL>", text)
    text = DATE_RE.sub("<DATE>", text)
    text = NUM_RE.sub("<NUM>", text)
    return text

def clean_corpus(df, column="content"):
    """Apply cleaning to the content column only."""
    df[column] = df[column].apply(clean_content)
    return df


# ── Tokenization ─────────────────────────────────────────────────────────────

def tokenize(text):
    """
    Tokenize a single string using regex.
    Extracts only lowercase alphabetic words, stripping punctuation automatically.
    ~20-50x faster than NLTK word_tokenize.
    """
    if not isinstance(text, str) or text == "":
        return []
    return TOKEN_RE.findall(text)

def tokenize_corpus(df, column="content"):
    """Tokenize per article, then drop the raw content column to save RAM."""
    df["tokens"] = df[column].apply(tokenize)
    del df[column]
    return df


# ── Stopword Removal ──────────────────────────────────────────────────────────

def remove_stopwords(tokens):
    """Remove stopwords from a list of tokens, guarding against NaN values."""
    return [t for t in tokens if isinstance(t, str) and t not in STOP_WORDS]

def remove_stopwords_corpus(df, column="tokens"):
    exploded = df[column].explode()
    filtered = exploded[~exploded.isin(STOP_WORDS)]
    df["tokens_nostop"] = filtered.groupby(level=0).agg(list)
    df["tokens_nostop"] = df["tokens_nostop"].apply(
        lambda x: x if isinstance(x, list) else []
    )
    del df[column]
    return df


# ── Stemming ──────────────────────────────────────────────────────────────────

def stem_tokens(tokens):
    """Apply Porter stemming to a list of tokens, guarding against NaN values."""
    return [STEMMER.stem(t) for t in tokens if isinstance(t, str)]

def stem_corpus(df, column="tokens_nostop"):
    """Apply stemming per article, dropping the pre-stem column to save RAM."""
    df["tokens_stemmed"] = df[column].apply(stem_tokens)
    del df[column]
    return df


# ── Vocabulary Stats ──────────────────────────────────────────────────────────

def vocab_stats(df):
    """Report vocabulary size at each processing stage."""
    stemmed      = set(t for tokens in df["tokens_stemmed"] for t in tokens)
    total_tokens = sum(len(t) for t in df["tokens_stemmed"])

    print(f"  Total token count:       {total_tokens:,}")
    print(f"  Vocabulary (stemmed):    {len(stemmed):,}")


# ── Train / Validation / Test Split ───────────────────────────────────────────

def split_data(df, train=0.8, val=0.1, test=0.1, seed=42):
    """Split DataFrame into train, validation, and test sets."""
    assert round(train + val + test, 10) == 1.0, "Splits must sum to 1."
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    n = len(df)
    train_end = int(n * train)
    val_end   = int(n * (train + val))
    return df[:train_end], df[train_end:val_end], df[val_end:]


# ── Chunked Pipeline ──────────────────────────────────────────────────────────

def process_chunk(chunk, column="content"):
    """Run the full pipeline on a single chunk."""
    chunk = clean_corpus(chunk, column)
    chunk = tokenize_corpus(chunk, column)
    chunk = remove_stopwords_corpus(chunk)
    chunk = stem_corpus(chunk)
    return chunk

def process_corpus(path, column="content", chunksize=10000):
    """
    Full pipeline in chunks: load → clean → tokenize → stopwords → stem.
    Processes chunksize rows at a time to keep RAM usage manageable.
    Returns the processed DataFrame and train/val/test splits.
    """
    csv.field_size_limit(sys.maxsize)
    chunks = []
    total_start = time.time()
    total_rows  = 0

    print("Starting chunked pipeline...")
    print_memory()

    for i, chunk in enumerate(pd.read_csv(path,
                                           encoding="utf-8",
                                           engine="python",
                                           chunksize=chunksize)):
        chunk = process_chunk(chunk, column)
        chunks.append(chunk)
        total_rows += len(chunk)

        if i % 10 == 0:
            elapsed = time.time() - total_start
            rate    = total_rows / elapsed if elapsed > 0 else 0
            eta     = (995000 - total_rows) / rate if rate > 0 else 0
            print(f"  Chunk {i+1:>4} | Rows: {total_rows:>8,} | "
                  f"Elapsed: {elapsed/60:.1f}m | ETA: {eta/60:.1f}m")
            print_memory()

    print("\nConcatenating chunks...")
    t  = time.time()
    df = pd.concat(chunks, ignore_index=True)
    del chunks
    print(f"  Done in {time.time()-t:.1f}s")
    print_memory()

    print("\nVocabulary stats:")
    vocab_stats(df)

    print("\nSplitting data...")
    train_df, val_df, test_df = split_data(df)
    print(f"  Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
    print(f"\nTotal time: {(time.time()-total_start)/60:.1f} minutes")

    return df, train_df, val_df, test_df

In [2]:
PATH = r"C:\Users\Kian\Documents\GitHub\GDSProject\995,000_rows.csv"
df, train_df, val_df, test_df = process_corpus(PATH)

Starting chunked pipeline...
  RAM — Python: 0.2 GB | Available: 9.3 GB
  Chunk    1 | Rows:   10,000 | Elapsed: 0.6m | ETA: 62.3m
  RAM — Python: 0.6 GB | Available: 8.9 GB
  Chunk   11 | Rows:  110,000 | Elapsed: 6.7m | ETA: 53.5m
  RAM — Python: 2.8 GB | Available: 6.8 GB
  Chunk   21 | Rows:  210,000 | Elapsed: 12.5m | ETA: 46.6m
  RAM — Python: 4.9 GB | Available: 5.7 GB
  Chunk   31 | Rows:  310,000 | Elapsed: 18.1m | ETA: 40.0m
  RAM — Python: 7.0 GB | Available: 3.5 GB
  Chunk   41 | Rows:  410,000 | Elapsed: 23.7m | ETA: 33.8m
  RAM — Python: 9.2 GB | Available: 1.5 GB
  Chunk   51 | Rows:  510,000 | Elapsed: 29.4m | ETA: 28.0m
  RAM — Python: 10.8 GB | Available: 0.8 GB
  Chunk   61 | Rows:  610,000 | Elapsed: 35.2m | ETA: 22.2m
  RAM — Python: 11.7 GB | Available: 0.6 GB
  Chunk   71 | Rows:  710,000 | Elapsed: 41.1m | ETA: 16.5m
  RAM — Python: 12.5 GB | Available: 0.5 GB
  Chunk   81 | Rows:  810,000 | Elapsed: 47.0m | ETA: 10.7m
  RAM — Python: 13.1 GB | Available: 1.0 GB

In [ ]:
def save_splits(train_df, val_df, test_df, output_dir=r"C:\Users\Kian\Documents\GitHub\GDSProject"):
    """Save train, validation, and test splits to pickle files."""
    import os

    splits = {
        "train": train_df,
        "val":   val_df,
        "test":  test_df
    }

    for name, split in splits.items():
        path = os.path.join(output_dir, f"{name}.pkl")
        split.to_pickle(path)
        print(f"Saved {name}: {len(split):,} rows → {path}")

save_splits(train_df, val_df, test_df)